<a href="https://colab.research.google.com/github/Shizukem/cu-i-k-/blob/main/b%C3%A0i_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np

projects = {
    1:  {"name": "TTDL Quốc gia Hòa Lạc",          "field": "Hạ tầng",       "C": 12000, "B": 21500, "C1": 8500, "C35": 3500},
    2:  {"name": "TTDL Quốc gia phía Nam",          "field": "Hạ tầng",       "C": 11500, "B": 20800, "C1": 7500, "C35": 4000},
    3:  {"name": "5G phủ sóng toàn quốc",           "field": "Hạ tầng",       "C": 18000, "B": 32500, "C1": 12000,"C35": 6000},
    4:  {"name": "VNeID 2.0",                        "field": "Chính phủ số",   "C": 4500,  "B": 9200,  "C1": 3500, "C35": 1000},
    5:  {"name": "Cổng DVC Quốc gia v3",            "field": "Chính phủ số",   "C": 3200,  "B": 6800,  "C1": 2500, "C35": 700},
    6:  {"name": "Y tế số quốc gia",                "field": "Y tế số",        "C": 5800,  "B": 11400, "C1": 4000, "C35": 1800},
    7:  {"name": "Giáo dục số K-12",                "field": "Giáo dục",       "C": 6500,  "B": 12200, "C1": 4500, "C35": 2000},
    8:  {"name": "TT AI Quốc gia + supercomputing", "field": "AI",             "C": 15000, "B": 28500, "C1": 9000, "C35": 6000},
    9:  {"name": "Sandbox tài chính số",            "field": "Tài chính số",   "C": 2500,  "B": 5800,  "C1": 1800, "C35": 700},
    10: {"name": "Logistics thông minh + cảng số",  "field": "Logistics",      "C": 7200,  "B": 13800, "C1": 5000, "C35": 2200},
    11: {"name": "Nông nghiệp số ĐBSCL",            "field": "Nông nghiệp",    "C": 4800,  "B": 8500,  "C1": 3500, "C35": 1300},
    12: {"name": "Đào tạo 50.000 kỹ sư AI/bán dẫn","field": "Nhân lực",       "C": 8500,  "B": 16200, "C1": 5500, "C35": 3000},
    13: {"name": "Khu CN bán dẫn Bắc Ninh-Bắc Giang","field": "Bán dẫn",     "C": 20000, "B": 35000, "C1": 13000,"C35": 7000},
    14: {"name": "An ninh mạng quốc gia (SOC)",     "field": "An ninh",        "C": 3800,  "B": 7500,  "C1": 2800, "C35": 1000},
    15: {"name": "Open Data Quốc gia",              "field": "Dữ liệu",        "C": 1500,  "B": 3800,  "C1": 1200, "C35": 300}
}

P  = list(range(1, 16))   # [1..15]
n  = len(P)                # 15 dự án

# Arrays theo thứ tự index 0..14 (dự án 1..15)
C_arr  = np.array([projects[i]["C"]  for i in P], dtype=float)  # chi phí tổng
B_arr  = np.array([projects[i]["B"]  for i in P], dtype=float)  # lợi ích NPV
C1_arr = np.array([projects[i]["C1"] for i in P], dtype=float)  # chi phí năm 1-2

# Tỷ suất lợi ích / chi phí (để tham chiếu)
BC_ratio = B_arr / C_arr

In [8]:
from scipy.optimize import milp, Bounds, LinearConstraint
import numpy as np

def print_result(y_sol, label="KẾT QUẢ"):
    """In danh sách dự án được chọn và các KPI."""
    selected = [i for i in range(n) if y_sol[i] > 0.5]
    proj_ids = [P[i] for i in selected]
    total_cost    = sum(C_arr[i]  for i in selected)
    total_benefit = sum(B_arr[i]  for i in selected)
    total_C1      = sum(C1_arr[i] for i in selected)

    print(f"\n{'='*62}")
    print(f"  {label}")
    print(f"{'='*62}")
    print(f"  Số dự án được chọn : {len(selected)}")
    print(f"  Tổng chi phí       : {total_cost:,.0f} tỷ VND")
    print(f"  Chi phí năm 1-2    : {total_C1:,.0f} tỷ VND")
    print(f"  Tổng lợi ích NPV   : {total_benefit:,.0f} tỷ VND")
    print(f"  NPV biên (B/C)     : {total_benefit/total_cost:.4f}")
    print(f"\n  {'Mã':<5} {'Tên dự án':<40} {'C (tỷ)':>8} {'B (tỷ)':>8} {'B/C':>6}")
    print(f"  {'-'*68}")
    for i in selected:
        p = P[i]
        print(f"  P{p:<4} {projects[p]['name']:<40} {C_arr[i]:>8,.0f} {B_arr[i]:>8,.0f} {BC_ratio[i]:>6.2f}")
    print(f"{'='*62}")
    return proj_ids, total_cost, total_benefit


def solve_mip(budget_total=80000, budget_12=40000,
              force_P1_P2=False, extra_constraints=None,
              use_expected_benefit=False):
    """
    Giải bài toán MIP theo cấu trúc đề bài mục 5.3.

    Tham số:
        budget_total   : Ngân sách 5 năm (mặc định 80.000 tỷ)
        budget_12      : Ngân sách năm 1-2 (mặc định 40.000 tỷ)
        force_P1_P2    : True => bắt buộc chọn cả P1 và P2 (câu 5.4.3)
        extra_constraints: danh sách hàng A_ub thêm [(row, ub), ...]
        use_expected_benefit: True => dùng lợi ích kỳ vọng (câu 5.4.4)
    """
    # max Σ Bᵢ·yᵢ  <=>  min -Σ Bᵢ·yᵢ
    if use_expected_benefit:
        # Xác suất hoàn thành đúng tiến độ pᵢ (câu 5.4.4)
        field_p = {
            "Hạ tầng":     0.85,
            "Chính phủ số":0.75,
            "AI":          0.65,
            "Bán dẫn":     0.65,
            "Y tế số":     0.80,
            "Giáo dục":    0.80,
            "Tài chính số":0.80,
            "Logistics":   0.80,
            "Nông nghiệp": 0.80,
            "Nhân lực":    0.80,
            "An ninh":     0.80,
            "Dữ liệu":     0.80,
        }
        p_arr = np.array([field_p[projects[i]["field"]] for i in P], dtype=float)
        obj_coef = -(p_arr * B_arr)   # tối đa E[Z] = Σ pᵢBᵢyᵢ
    else:
        obj_coef = -B_arr.copy()      # tối đa Σ Bᵢyᵢ

    # ── Ràng buộc bất đẳng thức: A_ub @ y <= b_ub ────────────────────
    A_rows = []
    b_vals = []

    # (C1) Ngân sách tổng 5 năm
    A_rows.append(C_arr.copy())
    b_vals.append(budget_total)

    # (C2) Ngân sách năm 1-2
    A_rows.append(C1_arr.copy())
    b_vals.append(budget_12)

    # (C3) Loại trừ: y₁ + y₂ ≤ 1 (chỉ 1 trong 2 trung tâm DL)
    row_c3 = np.zeros(n)
    row_c3[0] = 1; row_c3[1] = 1   # dự án 1 và 2 = index 0 và 1
    A_rows.append(row_c3)
    b_vals.append(1)

    # (C4) Tiên quyết y₈ ≤ y₁₂  =>  y₈ - y₁₂ ≤ 0
    row_c4 = np.zeros(n)
    row_c4[7] = 1; row_c4[11] = -1  # P8=idx7, P12=idx11
    A_rows.append(row_c4)
    b_vals.append(0)

    # (C5) Tiên quyết y₁₃ ≤ y₁₂  =>  y₁₃ - y₁₂ ≤ 0
    row_c5 = np.zeros(n)
    row_c5[12] = 1; row_c5[11] = -1  # P13=idx12, P12=idx11
    A_rows.append(row_c5)
    b_vals.append(0)

    # (C6) Cân đối lĩnh vực:
    # y₄ + y₅ ≥ 1  =>  -y₄ - y₅ ≤ -1
    row_c6a = np.zeros(n)
    row_c6a[3] = -1; row_c6a[4] = -1   # P4=idx3, P5=idx4
    A_rows.append(row_c6a)
    b_vals.append(-1)

    # y₁₄ ≥ 1  =>  -y₁₄ ≤ -1  (an ninh mạng bắt buộc)
    row_c6b = np.zeros(n)
    row_c6b[13] = -1  # P14=idx13
    A_rows.append(row_c6b)
    b_vals.append(-1)

    # (C7) Số lượng dự án: 7 ≤ Σyᵢ ≤ 11
    ones = np.ones(n)
    A_rows.append(-ones)     # -Σyᵢ ≤ -7
    b_vals.append(-7)
    A_rows.append(ones)      # Σyᵢ ≤ 11
    b_vals.append(11)

    # Ràng buộc câu 5.4.3: bắt buộc cả P1 và P2
    if force_P1_P2:
        # y₁ ≥ 1  =>  -y₁ ≤ -1
        row_p1 = np.zeros(n); row_p1[0] = -1
        A_rows.append(row_p1); b_vals.append(-1)
        # y₂ ≥ 1  =>  -y₂ ≤ -1
        row_p2 = np.zeros(n); row_p2[1] = -1
        A_rows.append(row_p2); b_vals.append(-1)

    # Ràng buộc bổ sung (nếu có)
    if extra_constraints:
        for (row, ub) in extra_constraints:
            A_rows.append(np.array(row, dtype=float))
            b_vals.append(ub)

    A_mat = np.array(A_rows, dtype=float)
    b_vec = np.array(b_vals, dtype=float)

    constraints = LinearConstraint(A_mat, lb=-np.inf, ub=b_vec)
    bounds      = Bounds(lb=np.zeros(n), ub=np.ones(n))
    integrality = np.ones(n, dtype=int)   # tất cả biến là integer (0 hoặc 1)

    res = milp(
        c            = obj_coef,
        constraints  = constraints,
        integrality  = integrality,
        bounds       = bounds,
    )
    return res

In [9]:
print("\n" + "█"*62)
print("  CÂU 5.4.1 — BÀI TOÁN GỐC (Ngân sách 80.000 tỷ VND)")
print("█"*62)

res1 = solve_mip(budget_total=80000, budget_12=40000)

if res1.status == 0:
    proj_ids_1, cost_1, benefit_1 = print_result(res1.x, "CÂU 5.4.1 — KẾT QUẢ TỐI ƯU")
    print(f"\n  ► Trạng thái solver : OPTIMAL (HiGHS)")
    print(f"  ► Z* (Tổng lợi ích) : {benefit_1:,.0f} tỷ VND")
    print(f"  ► Chi phí sử dụng   : {cost_1:,.0f} / 80.000 tỷ VND")
    print(f"  ► Dư ngân sách      : {80000-cost_1:,.0f} tỷ VND")
else:
    print(f"  ► Solver status: {res1.status} - {res1.message}")


██████████████████████████████████████████████████████████████
  CÂU 5.4.1 — BÀI TOÁN GỐC (Ngân sách 80.000 tỷ VND)
██████████████████████████████████████████████████████████████

  CÂU 5.4.1 — KẾT QUẢ TỐI ƯU
  Số dự án được chọn : 9
  Tổng chi phí       : 59,700 tỷ VND
  Chi phí năm 1-2    : 39,800 tỷ VND
  Tổng lợi ích NPV   : 115,400 tỷ VND
  NPV biên (B/C)     : 1.9330

  Mã    Tên dự án                                  C (tỷ)   B (tỷ)    B/C
  --------------------------------------------------------------------
  P2    TTDL Quốc gia phía Nam                     11,500   20,800   1.81
  P5    Cổng DVC Quốc gia v3                        3,200    6,800   2.12
  P7    Giáo dục số K-12                            6,500   12,200   1.88
  P8    TT AI Quốc gia + supercomputing            15,000   28,500   1.90
  P9    Sandbox tài chính số                        2,500    5,800   2.32
  P10   Logistics thông minh + cảng số              7,200   13,800   1.92
  P12   Đào tạo 50.000 kỹ sư AI/b

In [10]:
print("\n" + "█"*62)
print("  CÂU 5.4.2 — NÂNG NGÂN SÁCH LÊN 100.000 tỷ VND")
print("█"*62)

res2 = solve_mip(budget_total=100000, budget_12=50000)

if res2.status == 0:
    proj_ids_2, cost_2, benefit_2 = print_result(res2.x, "CÂU 5.4.2 — KẾT QUẢ (100.000 tỷ)")
    print(f"\n  ► Z* tăng thêm : +{benefit_2 - benefit_1:,.0f} tỷ VND")
    new_projs = set(proj_ids_2) - set(proj_ids_1)
    removed   = set(proj_ids_1) - set(proj_ids_2)
    if new_projs:
        names_new = [f"P{p} ({projects[p]['name']})" for p in sorted(new_projs)]
        print(f"  ► Dự án được thêm   : {', '.join(names_new)}")
    if removed:
        names_rem = [f"P{p} ({projects[p]['name']})" for p in sorted(removed)]
        print(f"  ► Dự án bị loại bỏ  : {', '.join(names_rem)}")
    if not new_projs and not removed:
        print("  ► Tập dự án không thay đổi (tất cả đã tối ưu trong 80.000 tỷ)")
else:
    print(f"  ► Solver status: {res2.status}")


██████████████████████████████████████████████████████████████
  CÂU 5.4.2 — NÂNG NGÂN SÁCH LÊN 100.000 tỷ VND
██████████████████████████████████████████████████████████████

  CÂU 5.4.2 — KẾT QUẢ (100.000 tỷ)
  Số dự án được chọn : 10
  Tổng chi phí       : 74,300 tỷ VND
  Chi phí năm 1-2    : 49,800 tỷ VND
  Tổng lợi ích NPV   : 142,500 tỷ VND
  NPV biên (B/C)     : 1.9179

  Mã    Tên dự án                                  C (tỷ)   B (tỷ)    B/C
  --------------------------------------------------------------------
  P2    TTDL Quốc gia phía Nam                     11,500   20,800   1.81
  P3    5G phủ sóng toàn quốc                      18,000   32,500   1.81
  P4    VNeID 2.0                                   4,500    9,200   2.04
  P5    Cổng DVC Quốc gia v3                        3,200    6,800   2.12
  P6    Y tế số quốc gia                            5,800   11,400   1.97
  P8    TT AI Quốc gia + supercomputing            15,000   28,500   1.90
  P9    Sandbox tài chính số   

In [11]:
from scipy.optimize import milp, Bounds, LinearConstraint
import numpy as np

# Project data (copied from first cell for scope)
projects = {
    1:  {"name": "TTDL Quốc gia Hòa Lạc",          "field": "Hạ tầng",       "C": 12000, "B": 21500, "C1": 8500, "C35": 3500},
    2:  {"name": "TTDL Quốc gia phía Nam",          "field": "Hạ tầng",       "C": 11500, "B": 20800, "C1": 7500, "C35": 4000},
    3:  {"name": "5G phủ sóng toàn quốc",           "field": "Hạ tầng",       "C": 18000, "B": 32500, "C1": 12000,"C35": 6000},
    4:  {"name": "VNeID 2.0",                        "field": "Chính phủ số",   "C": 4500,  "B": 9200,  "C1": 3500, "C35": 1000},
    5:  {"name": "Cổng DVC Quốc gia v3",            "field": "Chính phủ số",   "C": 3200,  "B": 6800,  "C1": 2500, "C35": 700},
    6:  {"name": "Y tế số quốc gia",                "field": "Y tế số",        "C": 5800,  "B": 11400, "C1": 4000, "C35": 1800},
    7:  {"name": "Giáo dục số K-12",                "field": "Giáo dục",       "C": 6500,  "B": 12200, "C1": 4500, "C35": 2000},
    8:  {"name": "TT AI Quốc gia + supercomputing", "field": "AI",             "C": 15000, "B": 28500, "C1": 9000, "C35": 6000},
    9:  {"name": "Sandbox tài chính số",            "field": "Tài chính số",   "C": 2500,  "B": 5800,  "C1": 1800, "C35": 700},
    10: {"name": "Logistics thông minh + cảng số",  "field": "Logistics",      "C": 7200,  "B": 13800, "C1": 5000, "C35": 2200},
    11: {"name": "Nông nghiệp số ĐBSCL",            "field": "Nông nghiệp",    "C": 4800,  "B": 8500,  "C1": 3500, "C35": 1300},
    12: {"name": "Đào tạo 50.000 kỹ sư AI/bán dẫn","field": "Nhân lực",       "C": 8500,  "B": 16200, "C1": 5500, "C35": 3000},
    13: {"name": "Khu CN bán dẫn Bắc Ninh-Bắc Giang","field": "Bán dẫn",     "C": 20000, "B": 35000, "C1": 13000,"C35": 7000},
    14: {"name": "An ninh mạng quốc gia (SOC)",     "field": "An ninh",        "C": 3800,  "B": 7500,  "C1": 2800, "C35": 1000},
    15: {"name": "Open Data Quốc gia",              "field": "Dữ liệu",        "C": 1500,  "B": 3800,  "C1": 1200, "C35": 300}
}

P  = list(range(1, 16))   # [1..15]
n  = len(P)                # 15 dự án

# Arrays theo thứ tự index 0..14 (dự án 1..15)
C_arr  = np.array([projects[i]["C"]  for i in P], dtype=float)  # chi phí tổng
B_arr  = np.array([projects[i]["B"]  for i in P], dtype=float)  # lợi ích NPV
C1_arr = np.array([projects[i]["C1"] for i in P], dtype=float)  # chi phí năm 1-2

# Tỷ suất lợi ích / chi phí (để tham chiếu)
BC_ratio = B_arr / C_arr

P_NAMES = [projects[p]["name"] for p in P]
C = C_arr # Alias for consistency with the problem description
B = B_arr # Alias for consistency with the problem description

def print_result(y_sol, label="KẾT QUẢ"):
    """In danh sách dự án được chọn và các KPI."""
    selected = [i for i in range(n) if y_sol[i] > 0.5]
    proj_ids = [P[i] for i in selected]
    total_cost    = sum(C_arr[i]  for i in selected)
    total_benefit = sum(B_arr[i]  for i in selected)
    total_C1      = sum(C1_arr[i] for i in selected)

    # The print statements were removed to make this a utility function
    # for internal use within the cell to define res1 for sol0 calculation.
    # If the user wishes to see the output for print_result, they can call it explicitly.

    return proj_ids, total_cost, total_benefit

def solve_mip(budget_total=80000, budget_12=40000,
              force_P1_P2=False, extra_constraints=None,
              use_expected_benefit=False):
    """
    Giải bài toán MIP theo cấu trúc đề bài mục 5.3.

    Tham số:
        budget_total   : Ngân sách 5 năm (mặc định 80.000 tỷ)
        budget_12      : Ngân sách năm 1-2 (mặc định 40.000 tỷ)
        force_P1_P2    : True => bắt buộc chọn cả P1 và P2 (câu 5.4.3)
        extra_constraints: danh sách hàng A_ub thêm [(row, ub), ...]
        use_expected_benefit: True => dùng lợi ích kỳ vọng (câu 5.4.4)
    """
    # max Σ Bᵢ·yᵢ  <=>  min -Σ Bᵢ·yᵢ
    if use_expected_benefit:
        # Xác suất hoàn thành đúng tiến độ pᵢ (câu 5.4.4)
        field_p = {
            "Hạ tầng":     0.85,
            "Chính phủ số":0.75,
            "AI":          0.65,
            "Bán dẫn":     0.65,
            "Y tế số":     0.80,
            "Giáo dục":    0.80,
            "Tài chính số":0.80,
            "Logistics":   0.80,
            "Nông nghiệp": 0.80,
            "Nhân lực":    0.80,
            "An ninh":     0.80,
            "Dữ liệu":     0.80,
        }
        p_arr = np.array([field_p[projects[i]["field"]] for i in P], dtype=float)
        obj_coef = -(p_arr * B_arr)   # tối đa E[Z] = Σ pᵢBᵢyᵢ
    else:
        obj_coef = -B_arr.copy()      # tối đa Σ Bᵢyᵢ

    # ── Ràng buộc bất đẳng thức: A_ub @ y <= b_ub ────────────────────
    A_rows = []
    b_vals = []

    # (C1) Ngân sách tổng 5 năm
    A_rows.append(C_arr.copy())
    b_vals.append(budget_total)

    # (C2) Ngân sách năm 1-2
    A_rows.append(C1_arr.copy())
    b_vals.append(budget_12)

    # (C3) Loại trừ: y₁ + y₂ ≤ 1 (chỉ 1 trong 2 trung tâm DL)
    row_c3 = np.zeros(n)
    row_c3[0] = 1; row_c3[1] = 1   # dự án 1 và 2 = index 0 và 1
    A_rows.append(row_c3)
    b_vals.append(1)

    # (C4) Tiên quyết y₈ ≤ y₁₂  =>  y₈ - y₁₂ ≤ 0
    row_c4 = np.zeros(n)
    row_c4[7] = 1; row_c4[11] = -1  # P8=idx7, P12=idx11
    A_rows.append(row_c4)
    b_vals.append(0)

    # (C5) Tiên quyết y₁₃ ≤ y₁₂  =>  y₁₃ - y₁₂ ≤ 0
    row_c5 = np.zeros(n)
    row_c5[12] = 1; row_c5[11] = -1  # P13=idx12, P12=idx11
    A_rows.append(row_c5)
    b_vals.append(0)

    # (C6) Cân đối lĩnh vực:
    # y₄ + y₅ ≥ 1  =>  -y₄ - y₅ ≤ -1
    row_c6a = np.zeros(n)
    row_c6a[3] = -1; row_c6a[4] = -1   # P4=idx3, P5=idx4
    A_rows.append(row_c6a)
    b_vals.append(-1)

    # y₁₄ ≥ 1  =>  -y₁₄ ≤ -1  (an ninh mạng bắt buộc)
    row_c6b = np.zeros(n)
    row_c6b[13] = -1  # P14=idx13
    A_rows.append(row_c6b)
    b_vals.append(-1)

    # (C7) Số lượng dự án: 7 ≤ Σyᵢ ≤ 11
    ones = np.ones(n)
    A_rows.append(-ones)     # -Σyᵢ ≤ -7
    b_vals.append(-7)
    A_rows.append(ones)      # Σyᵢ ≤ 11
    b_vals.append(11)

    # Ràng buộc câu 5.4.3: bắt buộc cả P1 và P2
    if force_P1_P2:
        # y₁ ≥ 1  =>  -y₁ ≤ -1
        row_p1 = np.zeros(n); row_p1[0] = -1
        A_rows.append(row_p1); b_vals.append(-1)
        # y₂ ≥ 1  =>  -y₂ ≤ -1
        row_p2 = np.zeros(n); row_p2[1] = -1
        A_rows.append(row_p2); b_vals.append(-1)

    # Ràng buộc bổ sung (nếu có)
    if extra_constraints:
        for (row, ub) in extra_constraints:
            A_rows.append(np.array(row, dtype=float))
            b_vals.append(ub)

    A_mat = np.array(A_rows, dtype=float)
    b_vec = np.array(b_vals, dtype=float)

    constraints = LinearConstraint(A_mat, lb=-np.inf, ub=b_vec)
    bounds      = Bounds(lb=np.zeros(n), ub=np.ones(n))
    integrality = np.ones(n, dtype=int)   # tất cả biến là integer (0 hoặc 1)

    res = milp(
        c            = obj_coef,
        constraints  = constraints,
        integrality  = integrality,
        bounds       = bounds,
    )
    return res

def parse_result(res):
    """
    Parses the result from the MILP solver and returns a dictionary of KPIs.
    """
    if res.status != 0: # Not optimal, potentially infeasible
        return None

    y_sol = res.x
    selected = [i for i in range(n) if y_sol[i] > 0.5]
    total_cost    = sum(C_arr[i]  for i in selected)
    total_benefit = sum(B_arr[i]  for i in selected)
    total_C1      = sum(C1_arr[i] for i in selected)
    n_projects    = len(selected)
    npv_per_cost  = total_benefit / total_cost if total_cost > 0 else 0

    return {
        'selected': selected,
        'total_cost': total_cost,
        'z_star': total_benefit,
        'n_projects': n_projects,
        'total_c1': total_C1,
        'npv_per_cost': npv_per_cost
    }

def build_and_solve(budget_total=80000, budget_12=40000,
                    fix_y1=False, fix_y2=False, require_p1p2=False,
                    extra_lb=None, extra_ub=None): # Added extra_lb and extra_ub
    """
    Builds and solves the MILP problem with flexible constraints for C3 and P1/P2.
    """
    obj_coef = -B_arr.copy() # Maximize sum(B*y) -> minimize sum(-B*y)

    A_rows = []
    b_vals = []

    # (C1) Ngân sách tổng 5 năm
    A_rows.append(C_arr.copy())
    b_vals.append(budget_total)

    # (C2) Ngân sách năm 1-2
    A_rows.append(C1_arr.copy())
    b_vals.append(budget_12)

    # (C3) Loại trừ: y₁ + y₂ ≤ 1 (chỉ 1 trong 2 trung tâm DL)
    # OR: relaxed to y₁ + y₂ ≤ 2 if require_p1p2 is True
    if require_p1p2:
        row_c3_relaxed = np.zeros(n)
        row_c3_relaxed[0] = 1; row_c3_relaxed[1] = 1
        A_rows.append(row_c3_relaxed)
        b_vals.append(2) # Relaxed upper bound
    else:
        row_c3 = np.zeros(n)
        row_c3[0] = 1; row_c3[1] = 1
        A_rows.append(row_c3)
        b_vals.append(1)

    # (C4) Tiên quyết y₈ ≤ y₁₂  =>  y₈ - y₁₂ ≤ 0
    row_c4 = np.zeros(n)
    row_c4[7] = 1; row_c4[11] = -1
    A_rows.append(row_c4)
    b_vals.append(0)

    # (C5) Tiên quyết y₁₃ ≤ y₁₂  =>  y₁₃ - y₁₂ ≤ 0
    row_c5 = np.zeros(n)
    row_c5[12] = 1; row_c5[11] = -1
    A_rows.append(row_c5)
    b_vals.append(0)

    # (C6) Cân đối lĩnh vực:
    # y₄ + y₅ ≥ 1  =>  -y₄ - y₅ ≤ -1
    row_c6a = np.zeros(n)
    row_c6a[3] = -1; row_c6a[4] = -1
    A_rows.append(row_c6a)
    b_vals.append(-1)

    # y₁₄ ≥ 1  =>  -y₁₄ ≤ -1  (an ninh mạng bắt buộc)
    row_c6b = np.zeros(n)
    row_c6b[13] = -1
    A_rows.append(row_c6b)
    b_vals.append(-1)

    # (C7) Số lượng dự án: 7 ≤ Σyᵢ ≤ 11
    ones = np.ones(n)
    A_rows.append(-ones)     # -Σyᵢ ≤ -7
    b_vals.append(-7)
    A_rows.append(ones)      # Σyᵢ ≤ 11
    b_vals.append(11)

    # Specific constraints for forcing P1 and P2
    if fix_y1:
        row_y1 = np.zeros(n); row_y1[0] = -1
        A_rows.append(row_y1); b_vals.append(-1) # y1 >= 1
    if fix_y2:
        row_y2 = np.zeros(n); row_y2[1] = -1
        A_rows.append(row_y2); b_vals.append(-1) # y2 >= 1

    if require_p1p2:
        # These are already handled by the require_p1p2 logic when modifying C3
        # but explicitly adding them here for clarity if they were to be separate
        # from the C3 relaxation logic.
        row_p1 = np.zeros(n); row_p1[0] = -1
        A_rows.append(row_p1); b_vals.append(-1) # y1 >= 1
        row_p2 = np.zeros(n); row_p2[1] = -1
        A_rows.append(row_p2); b_vals.append(-1) # y2 >= 1


    A_mat = np.array(A_rows, dtype=float)
    b_vec = np.array(b_vals, dtype=float)

    constraints = LinearConstraint(A_mat, lb=-np.inf, ub=b_vec)

    # Apply extra_lb and extra_ub to bounds
    lb = np.zeros(n)
    ub = np.ones(n)
    if extra_lb is not None:
        lb = np.maximum(lb, extra_lb)
    if extra_ub is not None:
        ub = np.minimum(ub, extra_ub)

    bounds      = Bounds(lb=lb, ub=ub) # Modified bounds
    integrality = np.ones(n, dtype=int)

    res = milp(
        c            = obj_coef,
        constraints  = constraints,
        integrality  = integrality,
        bounds       = bounds,
    )
    return res

# Calculate res1 and sol0 based on res1
res1 = solve_mip(budget_total=80000, budget_12=40000)
sol0 = parse_result(res1)

print("\n" + "─"*72)
print("CÂU 5.4.3 — Quốc hội yêu cầu PHẢI CÓ CẢ P1 VÀ P2 (redundancy)")
print("─"*72)

# --- Bước 1: Kiểm tra với C3 nguyên vẹn ---
print("\n  BƯỚC 1: Kiểm tra khả thi khi GIỮ NGUYÊN C3 (y₁ + y₂ ≤ 1)")
print(f"  Ràng buộc C3 gốc: y₁ + y₂ ≤ 1 (chỉ chọn 1 địa điểm)")
print(f"  Yêu cầu mới     : y₁ = 1 VÀ y₂ = 1  →  y₁ + y₂ = 2 > 1")
print(f"  => C3 MU THUᔎN với yêu cầu mới → BI TON KHNG KHኆ THI ✗")

# Xác nhận bằng solver
res3_infeas = build_and_solve(budget_total=80_000, fix_y1=True, fix_y2=True)
print(f"\n  Xác nhận bằng solver (fix y₁=y₂=1, C3 y₁+y₂≤1):\n")
# The message `INFEASIBLE ✗ (đúng như phân tích)` is for res3_infeas.status == 2 (infeasible)
# The message `Lᕷ logic` is for other statuses (e.g. status == 0, meaning feasible, which would be an error in logic).
if res3_infeas.status == 2:
    print("  → Status = 2: INFEASIBLE ✗ (đúng như phân tích)")
else:
    print(f"  → Status = {res3_infeas.status}: Lᕷ logic")


# --- Bước 2: Nới C3 → bỏ ràng buộc loại trừ ---
print("\n  BƯỚ C 2: Nớ i C3 → cho phép cả hai (y₁ + y₂ ≤ 2)")
print("  Giả định: Quốc hội ghi đè C3 vì lý do dự phòng (redundancy)")
print("  C3 mới  : y₁ + y₂ ≤ 2 (tức là không còn ràng buộc loại trừ)")
print("  Đồng thời ép: y₁ = 1, y₂ = 1")

res3 = build_and_solve(budget_total=80_000, require_p1p2=True)
sol3 = parse_result(res3)

if sol3 is None:
    print("  → Vẫn INFEASIBLE sau khi nới C3!")
else:
    print(f"\n  → BI TON KHኆ THI sau khi nới C3 ✓")
    print(f"\n  {'Mã':<6} {'Tân dự án':<35} {'Chi phí':>10} {'NPV':>10} {'Trạng thái'}")
    print("  " + "─"*75)
    # So sánh với base case
    added   = [i for i in sol3['selected'] if i not in sol0['selected']]
    removed = [i for i in sol0['selected'] if i not in sol3['selected']]
    for i in sol3['selected']:
        status = "◀ THÊM (bắt buộc)" if i == 0 or i == 1 else "  giữ lại"
        print(f"  P{i+1:<5} {P_NAMES[i]:<35} {C[i]:>10,.0f} {B[i]:>10,.0f}   {status}")
    print("  " + "─"*75)
    print(f"  {'TỔNG':<41} {sol3['total_cost']:>10,.0f} {sol3['z_star']:>10,.0f}")

    # Bị loại
    print(f"\n  Dự án bị loại khỉi danh sách (so vớ i nghiệm cơ bản):")
    for i in removed:
        print(f"    ✘ P{i+1}: {P_NAMES[i]} (NPV={B[i]:,.0f}, C={C[i]:,.0f})")

    print(f"\n  {'─'*55}")
    print(f"  Chỉ tiâu              │  Cơ bản (5.4.1)  │  P1+P2 (5.4.3)")
    print(f"  {'─'*55}")
    print(f"  Z* (tỷ VND)           │  {sol0['z_star']:>14,.0f}  │  {sol3['z_star']:>14,.0f}")
    print(f"  Thay đổi Z*           │        —          │  {sol3['z_star']-sol0['z_star']:>+14,.0f}")
    print(f"  Thay đổi (% )          │        —          │  {(sol3['z_star']-sol0['z_star'])/sol0['z_star']*100:>+14.2f}%")
    print(f"  Số dự án              │  {sol0['n_projects']:>14}  │  {sol3['n_projects']:>14}")
    print(f"  Tổng chi phí          │  {sol0['total_cost']:>14,.0f}  │  {sol3['total_cost']:>14,.0f}")
    print(f"  C1 (năm 1-2)          │  {sol0['total_c1']:>14,.0f}  │  {sol3['total_c1']:>14,.0f}")
    print(f"  NPV biân (Z*/cost)    │  {sol0['npv_per_cost']:>14.4f}  │  {sol3['npv_per_cost']:>14.4f}")
    print(f"  {'─'*55}")

    delta_z = sol3['z_star'] - sol0['z_star']
    print(f"\n  KẶT LUẬN:")
    print(f"  • Z* giảm {abs(delta_z):,.0f} tỷ VND ({abs(delta_z)/sol0['z_star']*100:.2f}%)iesią");
    print(f"  • Lý do: Ép P1+P2 (chi phí {C[0]+C[1]:,.0f} tỷ) → phải bỏ các dự án")
    print(f"    có tổng NPV cao hơn ({sum(B[i] for i in removed):,.0f} tỷ) để đổi lấy")
    print(f"    P1+P2 (tổng NPV = {B[0]+B[1]:,.0f} tỷ)")
    print(f"  • Chi phí chính sách của redundancy: {abs(delta_z):,.0f} tỷ VND NPV")


────────────────────────────────────────────────────────────────────────
CÂU 5.4.3 — Quốc hội yêu cầu PHẢI CÓ CẢ P1 VÀ P2 (redundancy)
────────────────────────────────────────────────────────────────────────

  BƯỚC 1: Kiểm tra khả thi khi GIỮ NGUYÊN C3 (y₁ + y₂ ≤ 1)
  Ràng buộc C3 gốc: y₁ + y₂ ≤ 1 (chỉ chọn 1 địa điểm)
  Yêu cầu mới     : y₁ = 1 VÀ y₂ = 1  →  y₁ + y₂ = 2 > 1
  => C3 MU THUᔎN với yêu cầu mới → BI TON KHNG KHኆ THI ✗

  Xác nhận bằng solver (fix y₁=y₂=1, C3 y₁+y₂≤1):

  → Status = 2: INFEASIBLE ✗ (đúng như phân tích)

  BƯỚ C 2: Nớ i C3 → cho phép cả hai (y₁ + y₂ ≤ 2)
  Giả định: Quốc hội ghi đè C3 vì lý do dự phòng (redundancy)
  C3 mới  : y₁ + y₂ ≤ 2 (tức là không còn ràng buộc loại trừ)
  Đồng thời ép: y₁ = 1, y₂ = 1

  → BI TON KHኆ THI sau khi nới C3 ✓

  Mã     Tân dự án                              Chi phí        NPV Trạng thái
  ───────────────────────────────────────────────────────────────────────────
  P1     TTDL Quốc gia Hòa Lạc                   12,00

In [12]:
print("\n" + "█"*62)
print("  CÂU 5.4.4 — TỐI ĐA HÓA LỢI ÍCH KỲ VỌNG E[Z]")
print("█"*62)

# Xác suất hoàn thành đúng tiến độ
field_p = {
    "Hạ tầng":     0.85,
    "Chính phủ số":0.75,
    "AI":          0.65,
    "Bán dẫn":     0.65,
    "Y tế số":     0.80,
    "Giáo dục":    0.80,
    "Tài chính số":0.80,
    "Logistics":   0.80,
    "Nông nghiệp": 0.80,
    "Nhân lực":    0.80,
    "An ninh":     0.80,
    "Dữ liệu":     0.80,
}
p_arr = np.array([field_p[projects[i]["field"]] for i in P])
EB_arr = p_arr * B_arr

print("\n  Xác suất hoàn thành theo lĩnh vực:")
print(f"  {'Lĩnh vực':<18} {'p':>6} {'B (tỷ)':>10} {'E[B]=p×B':>12}")
print(f"  {'-'*48}")
for i in range(n):
    p_id = P[i]
    print(f"  P{p_id:<4} {projects[p_id]['field']:<18} {p_arr[i]:>6.2f} {B_arr[i]:>10,.0f} {EB_arr[i]:>12,.0f}")

res4 = solve_mip(budget_total=80000, budget_12=40000, use_expected_benefit=True)

if res4.status == 0:
    proj_ids_4, cost_4, _ = print_result(res4.x, "CÂU 5.4.4 — TỐI ĐA E[Z]")
    # Tính E[Z] thực tế
    selected4 = [i for i in range(n) if res4.x[i] > 0.5]
    EZ_val = sum(EB_arr[i] for i in selected4)
    Z_det  = sum(B_arr[i]  for i in selected4)
    print(f"\n  ► E[Z*] (lợi ích kỳ vọng)   : {EZ_val:,.0f} tỷ VND")
    print(f"  ► Z (lợi ích danh nghĩa)    : {Z_det:,.0f} tỷ VND")
    print(f"  ► Mức chiết khấu rủi ro     : {(1 - EZ_val/Z_det)*100:.1f}%")
    # So sánh với câu 5.4.1
    diff_44 = set(proj_ids_4) - set(proj_ids_1)
    rem_44  = set(proj_ids_1) - set(proj_ids_4)
    if diff_44:
        print(f"  ► Thêm dự án (vs 5.4.1): {', '.join([f'P{p}' for p in sorted(diff_44)])}")
    if rem_44:
        print(f"  ► Bỏ dự án  (vs 5.4.1): {', '.join([f'P{p}' for p in sorted(rem_44)])}")
    if not diff_44 and not rem_44:
        print("  ► Tập dự án giống 5.4.1 (rủi ro không thay đổi thứ hạng ưu tiên)")


██████████████████████████████████████████████████████████████
  CÂU 5.4.4 — TỐI ĐA HÓA LỢI ÍCH KỲ VỌNG E[Z]
██████████████████████████████████████████████████████████████

  Xác suất hoàn thành theo lĩnh vực:
  Lĩnh vực                p     B (tỷ)     E[B]=p×B
  ------------------------------------------------
  P1    Hạ tầng              0.85     21,500       18,275
  P2    Hạ tầng              0.85     20,800       17,680
  P3    Hạ tầng              0.85     32,500       27,625
  P4    Chính phủ số         0.75      9,200        6,900
  P5    Chính phủ số         0.75      6,800        5,100
  P6    Y tế số              0.80     11,400        9,120
  P7    Giáo dục             0.80     12,200        9,760
  P8    AI                   0.65     28,500       18,525
  P9    Tài chính số         0.80      5,800        4,640
  P10   Logistics            0.80     13,800       11,040
  P11   Nông nghiệp          0.80      8,500        6,800
  P12   Nhân lực             0.80     16,200    